# 0. Setup check

**What it does:** checks that everything the pipeline needs is in place before you run anything:
the Python packages, the SDH extraction folder, and the output folder. It reads nothing but file
names and row counts.

**Run:** once, whenever you start on a new machine or environment.
**Time:** seconds.

> **Privacy:** everything this notebook *displays* comes from `aggregate/` (counts with cells under 6 suppressed). Files in `patient_level/` stay on GEMINI: don't print their rows, and never paste them outside GEMINI.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# To write the output somewhere else, uncomment and edit BEFORE importing sdh_config:
# os.environ["SDH_OUTPUT_DIR"] = str(Path.home() / "sdh_phenotype_output")

PHENO = Path.cwd().parent              # the phenotype/ folder (this notebook is in phenotype/notebooks/)
sys.path.insert(0, str(PHENO))
import sdh_config as cfg
import sdh_io as io
pd.set_option("display.max_rows", 200, "display.max_columns", 60, "display.width", 220)
AGG = cfg.AGGREGATE_DIR


def show(name, rows=None, note=None):
    """Display one aggregate (cell-suppressed) table from the output folder."""
    path = AGG / name
    if not path.exists():
        print(f"{name}: not there yet (run the step above first)")
        return None
    df = pd.read_csv(path)
    display(Markdown(f"**{name}**" + (f" - {note}" if note else "")))
    display(df.head(rows) if rows else df)
    return df


def share(names, rows=80):
    """Print aggregate tables as text to paste to Claude / your team (counts only, small cells suppressed)."""
    for name in names:
        path = AGG / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"===== {name} ({len(df)} rows{', first ' + str(rows) if len(df) > rows else ''}) =====")
            print(df.head(rows).to_csv(index=False))
        else:
            print(f"===== {name}: not found =====")

## Packages

In [ ]:
import importlib
for pkg in ["pandas", "pyarrow", "spacy"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:8s} OK  {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"{pkg:8s} MISSING -> in a terminal: pip install {pkg}")
print("python  ", sys.version.split()[0], "|", sys.executable)

## Input: the GEMINI SDH extraction

In [ ]:
import pyarrow.parquet as pq
print("Data folder:", cfg.DATA_DIR, "| exists:", cfg.DATA_DIR.exists())
for f in ["cohort.parquet", "encounters.parquet", "sdh_surgery_codes.csv"]:
    print(f"  {f:28s}", "found" if (cfg.DATA_DIR / f).exists() else "MISSING")
need = ["radiology", "ipdiagnosis", "erdiagnosis", "ipintervention", "erintervention", "lab", "vitals",
        "pharmacy", "ipscu", "erconsults", "derived_variables", "lookup_cci", "lookup_cihi_codes",
        "lookup_lab_concept", "lookup_vitals_concept"]
for t in need:
    p = cfg.DATA_DIR / "raw" / f"{t}.parquet"
    print(f"  raw/{t:24s}", f"{pq.ParquetFile(p).metadata.num_rows:>12,} rows" if p.exists() else "MISSING")

## Output folder

In [ ]:
cfg.PATIENT_DIR.mkdir(parents=True, exist_ok=True)
cfg.AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)
print("Patient-level output (stays on GEMINI):", cfg.PATIENT_DIR)
print("Aggregate output (safe to share):      ", cfg.AGGREGATE_DIR)

**Look for:** every line says OK / found. If `spacy` is missing, install it before notebook 2.